# Modelo de probabilidade de NORM para o próximo mês

Este notebook usa os dados de um mês para estimar a probabilidade de aparecer **borra oleosa com NORM no mês seguinte**.

O código foi dividido em etapas pequenas e comentadas. Assim, fica mais fácil entender de onde vêm os dados, como as variáveis são criadas e como o modelo é avaliado.

## 1. Importar as bibliotecas

Bibliotecas são conjuntos de ferramentas prontas. Aqui usamos ferramentas para ler tabelas, fazer cálculos, criar gráficos e treinar modelos de aprendizado de máquina.

In [ ]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## 2. Ler os dados

O caminho é montado a partir da pasta do projeto. Dessa forma, o notebook funciona mesmo quando o projeto é movido para outro computador.

Também definimos os nomes que serão usados várias vezes ao longo do código.

In [ ]:
# Se o notebook estiver sendo executado dentro da pasta `notebooks`,
# voltamos uma pasta para encontrar a raiz do projeto.
PASTA_PROJETO = Path.cwd()
if not (PASTA_PROJETO / 'data').exists():
    PASTA_PROJETO = PASTA_PROJETO.parent

PASTA_DADOS = PASTA_PROJETO / 'data' / 'processed'

COLUNA_PLATAFORMA = 'LOCAL DA GERAÇÃO'
COLUNA_TIPO_RESIDUO = 'TIPO DE RESÍDUO'
COLUNA_ALVO_ATUAL = 'TEM_NORM_MES'
COLUNA_ALVO_FUTURO = 'TEM_NORM_PROXIMO_MES'

VARIAVEIS_QUIMICAS = ['SALINIDADE', 'BARIO', 'ESTRONCIO']
PRECISAO_MINIMA = 0.50
PLATAFORMA_ANALISE = 'P-58'

dados_fenix = pd.read_csv(PASTA_DADOS / 'fenix_processado.csv')
dados_residuos = pd.read_csv(PASTA_DADOS / 'base_integrada_residuos.csv')

# Algumas bases antigas podem ter os títulos das colunas com problema de codificação.
# Este trecho encontra os nomes equivalentes automaticamente.
if COLUNA_PLATAFORMA not in dados_residuos.columns:
    COLUNA_PLATAFORMA = 'LOCAL DA GERAÇÃO'
if COLUNA_TIPO_RESIDUO not in dados_residuos.columns:
    COLUNA_TIPO_RESIDUO = 'TIPO DE RESÃDUO'

CHAVES = [COLUNA_PLATAFORMA, 'Mes']

for tabela in (dados_fenix, dados_residuos):
    tabela['Mes'] = pd.to_datetime(tabela['Mes'], errors='coerce')

print('Linhas da base Fênix:', len(dados_fenix))
print('Linhas da base de resíduos:', len(dados_residuos))

## 3. Criar a resposta que o modelo deve aprender

O modelo precisa de exemplos com resposta conhecida. Para cada plataforma e mês:

- `0` significa borra oleosa **sem NORM**;
- `1` significa borra oleosa **com NORM**.

Se houver mais de um registro no mesmo mês, usamos o maior valor. Portanto, basta existir um registro com NORM para o mês receber o valor 1.

In [ ]:
tipo_residuo = dados_residuos[COLUNA_TIPO_RESIDUO].astype('string').str.strip()

eh_borra_oleosa = tipo_residuo.str.contains('BORRA OLEOSA', case=False, na=False)
eh_borra_com_norm = tipo_residuo.str.contains(
    'BORRA OLEOSA COM NORM', case=False, na=False
)

rotulos = dados_residuos.loc[eh_borra_oleosa, CHAVES].copy()
rotulos[COLUNA_ALVO_ATUAL] = eh_borra_com_norm.loc[rotulos.index].astype(int)

rotulos = (
    rotulos.groupby(CHAVES, as_index=False)[COLUNA_ALVO_ATUAL]
    .max()
)

display(
    rotulos[COLUNA_ALVO_ATUAL]
    .value_counts()
    .rename('quantidade')
    .to_frame()
)

## 4. Escolher as variáveis de entrada

As variáveis de entrada são as informações que o modelo usa para calcular a probabilidade de NORM. Usamos Bário, Estrôncio, Salinidade, Sulfato e as quatro variáveis operacionais disponíveis.

In [ ]:
# O sulfato está no arquivo Fênix completo, antes da agregação mensal.
caminho_fenix_completo = PASTA_DADOS / 'fenix_compilado_todas_colunas.csv'
cabecalho_completo = pd.read_csv(caminho_fenix_completo, nrows=0).columns
COLUNA_SULFATO_ORIGINAL = next(
    coluna for coluna in cabecalho_completo
    if '[SULFATO]' in coluna.upper()
)

sulfato_bruto = pd.read_csv(
    caminho_fenix_completo,
    usecols=['SIGLA', 'DATA', 'POCO', 'QW_M3D', COLUNA_SULFATO_ORIGINAL],
)
sulfato_bruto['Mes'] = (
    pd.to_datetime(sulfato_bruto['DATA'], errors='coerce', format='mixed')
    .dt.to_period('M')
    .dt.to_timestamp()
)
sulfato_bruto['SULFATO'] = pd.to_numeric(
    sulfato_bruto[COLUNA_SULFATO_ORIGINAL], errors='coerce'
)
sulfato_bruto['QW_M3D'] = pd.to_numeric(
    sulfato_bruto['QW_M3D'], errors='coerce'
)

# Primeiro, obtemos uma mediana para cada poço em cada mês.
sulfato_por_poco = (
    sulfato_bruto
    .groupby(['SIGLA', 'Mes', 'POCO'], as_index=False)
    .agg(SULFATO=('SULFATO', 'median'), QW_M3D=('QW_M3D', 'median'))
)

def media_ponderada_sulfato(grupo):
    valores_validos = grupo.loc[
        (grupo['SULFATO'] > 0) & (grupo['QW_M3D'] > 0),
        ['SULFATO', 'QW_M3D'],
    ].dropna()

    if valores_validos.empty:
        return np.nan

    return np.average(
        valores_validos['SULFATO'],
        weights=valores_validos['QW_M3D'],
    )

sulfato_mensal = (
    sulfato_por_poco
    .groupby(['SIGLA', 'Mes'])
    .apply(media_ponderada_sulfato, include_groups=False)
    .rename('SULFATO')
    .reset_index()
    .rename(columns={'SIGLA': COLUNA_PLATAFORMA})
)

dados_fenix = dados_fenix.merge(
    sulfato_mensal, on=[COLUNA_PLATAFORMA, 'Mes'], how='left'
)

variaveis_operacionais = ['QW_M3D', 'QO_M3D', 'BSW', 'N_POCOS']
variaveis_desejadas = VARIAVEIS_QUIMICAS + variaveis_operacionais + ['SULFATO']

# Mantemos apenas as colunas encontradas no arquivo.
variaveis_atuais = [
    coluna for coluna in variaveis_desejadas
    if coluna in dados_fenix.columns
]

dados = dados_fenix[CHAVES + variaveis_atuais].copy()
dados = dados.sort_values(CHAVES).reset_index(drop=True)

# Converte as variáveis para número. Valores inválidos se tornam ausentes (NaN).
dados[variaveis_atuais] = dados[variaveis_atuais].apply(
    pd.to_numeric, errors='coerce'
)

print('Colunas disponíveis para preparar a comparação:')
print(variaveis_atuais)

## 5. Criar informações sobre os meses anteriores

Para Bário, Estrôncio e Salinidade, criamos o valor anterior (`LAG1`) e as médias dos 3 e 6 registros anteriores. Essas variáveis mostram ao modelo o histórico químico recente. As variáveis operacionais entram somente com seus valores atuais.

O Sulfato entra com seu valor do mês atual. Todas essas informações do mês `t` são usadas para prever a ocorrência de NORM no mês `t+1`.

In [ ]:
variaveis_historicas = []

for coluna in VARIAVEIS_QUIMICAS:
    # Separa o histórico por plataforma para não misturar locais diferentes.
    grupo = dados.groupby(COLUNA_PLATAFORMA, sort=False)[coluna]

    nome_lag1 = f'{coluna}_LAG1'
    nome_media_3m = f'{coluna}_MEDIA_3M'
    nome_media_6m = f'{coluna}_MEDIA_6M'

    dados[nome_lag1] = grupo.shift(1)
    dados[nome_media_3m] = grupo.transform(
        lambda serie: serie.shift(1).rolling(3, min_periods=2).mean()
    )
    dados[nome_media_6m] = grupo.transform(
        lambda serie: serie.shift(1).rolling(6, min_periods=3).mean()
    )

    variaveis_historicas.extend([
        nome_lag1, nome_media_3m, nome_media_6m
    ])

print('Variáveis históricas criadas:')
print(variaveis_historicas)

## 6. Ligar os dados atuais ao resultado do mês seguinte

O resultado observado em fevereiro, por exemplo, passa a ser a resposta da linha de janeiro. Assim, o modelo aprende a prever um mês à frente.

In [ ]:
rotulos_futuros = rotulos.rename(
    columns={COLUNA_ALVO_ATUAL: COLUNA_ALVO_FUTURO}
).copy()

# Move o rótulo um mês para trás para ligá-lo aos dados usados na previsão.
rotulos_futuros['Mes'] = (
    rotulos_futuros['Mes'] - pd.offsets.MonthBegin(1)
)

dados = dados.merge(rotulos_futuros, on=CHAVES, how='left')
dados['Mes_alvo'] = dados['Mes'] + pd.offsets.MonthBegin(1)

# Junta valores atuais, históricos, operacionais e Sulfato.
FEATURES = variaveis_atuais + variaveis_historicas

print('Quantidade total de variáveis usadas pelo modelo:', len(FEATURES))

## 7. Preparar a base de modelagem

Mantemos somente linhas com valores químicos válidos e com resposta conhecida para o mês seguinte. Valores ausentes nas outras variáveis serão preenchidos automaticamente pelo modelo.

In [ ]:
quimica_nao_nula = dados[VARIAVEIS_QUIMICAS].notna().all(axis=1)
quimica_maior_zero = (dados[VARIAVEIS_QUIMICAS] > 0).all(axis=1)
quimica_valida = quimica_nao_nula & quimica_maior_zero

possui_alvo_futuro = dados[COLUNA_ALVO_FUTURO].notna()

base = dados.loc[quimica_valida & possui_alvo_futuro].copy()
base = base.sort_values('Mes').reset_index(drop=True)
base[COLUNA_ALVO_FUTURO] = base[COLUNA_ALVO_FUTURO].astype(int)

print('Quantidade de linhas:', len(base))
print('Quantidade de plataformas:', base[COLUNA_PLATAFORMA].nunique())
display(
    base[COLUNA_ALVO_FUTURO]
    .value_counts()
    .rename('quantidade')
    .to_frame()
)

## 8. Separar treino, validação e teste por tempo

Não embaralhamos os meses, porque isso permitiria ao modelo aprender com o futuro. Usamos:

- os primeiros 70% dos meses para treino;
- os 15% seguintes para validação;
- os últimos 15% para teste final.

A validação escolhe o modelo e o limite de decisão. O teste fica reservado para a avaliação final.

In [ ]:
datas = np.array(sorted(base['Mes'].unique()))

if len(datas) < 3:
    raise ValueError('São necessários pelo menos 3 meses para dividir a base.')

indice_70 = int(len(datas) * 0.70) - 1
indice_85 = int(len(datas) * 0.85) - 1

corte_treino = pd.Timestamp(datas[indice_70])
corte_validacao = pd.Timestamp(datas[indice_85])

treino = base.loc[base['Mes'] <= corte_treino].copy()
validacao = base.loc[
    (base['Mes'] > corte_treino) &
    (base['Mes'] <= corte_validacao)
].copy()
teste = base.loc[base['Mes'] > corte_validacao].copy()

for nome, parte in {
    'Treino': treino,
    'Validação': validacao,
    'Teste': teste,
}.items():
    print(
        f'{nome}: {len(parte)} linhas, '
        f'{parte[COLUNA_PLATAFORMA].nunique()} plataformas, '
        f'{parte["Mes"].min():%Y-%m} até {parte["Mes"].max():%Y-%m}'
    )

## 9. Criar e comparar dois modelos

Vamos comparar regressão logística e Random Forest.

O preenchimento pela mediana trata valores ausentes. A regressão também precisa colocar as variáveis em escalas comparáveis. O balanceamento ajuda quando existem bem menos casos com NORM do que sem NORM.

In [ ]:
def criar_modelos():
    modelo_logistico = Pipeline([
        ('imputacao', SimpleImputer(strategy='median')),
        ('escala', StandardScaler()),
        ('modelo', LogisticRegression(
            class_weight='balanced',
            max_iter=5000,
            random_state=42,
        )),
    ])

    modelo_random_forest = Pipeline([
        ('imputacao', SimpleImputer(strategy='median')),
        ('modelo', RandomForestClassifier(
            n_estimators=500,
            min_samples_leaf=5,
            max_features='sqrt',
            class_weight='balanced_subsample',
            random_state=42,
            n_jobs=-1,
        )),
    ])

    return {
        'Regressão logística': modelo_logistico,
        'Random Forest': modelo_random_forest,
    }


comparacao = []
modelos_ajustados = {}

for nome, modelo in criar_modelos().items():
    modelo.fit(treino[FEATURES], treino[COLUNA_ALVO_FUTURO])
    probabilidades = modelo.predict_proba(validacao[FEATURES])[:, 1]

    comparacao.append({
        'MODELO': nome,
        'AUC_ROC': roc_auc_score(
            validacao[COLUNA_ALVO_FUTURO], probabilidades
        ),
        'AUC_PR': average_precision_score(
            validacao[COLUNA_ALVO_FUTURO], probabilidades
        ),
    })
    modelos_ajustados[nome] = (modelo, probabilidades)

comparacao = (
    pd.DataFrame(comparacao)
    .sort_values(['AUC_PR', 'AUC_ROC'], ascending=False)
    .reset_index(drop=True)
)

# AUC PR é a medida principal porque o evento com NORM é menos frequente.
nome_modelo = comparacao.loc[0, 'MODELO']
prob_validacao = modelos_ajustados[nome_modelo][1]

display(comparacao.round(3))
print('Modelo escolhido:', nome_modelo)

## 10. Escolher o limite de decisão

O modelo devolve uma probabilidade entre 0 e 1. Precisamos de um limite para transformar essa probabilidade em alerta.

Testamos vários limites e escolhemos o que encontra mais casos com NORM (`recall`), desde que pelo menos 50% dos alertas estejam corretos (`precisão`). Esse mínimo pode ser alterado no início do notebook.

In [ ]:
resultados_limiares = []

for limiar in np.linspace(0.01, 0.99, 981):
    previsao = (prob_validacao >= limiar).astype(int)

    resultados_limiares.append({
        'LIMIAR': limiar,
        'PRECISAO': precision_score(
            validacao[COLUNA_ALVO_FUTURO], previsao, zero_division=0
        ),
        'RECALL': recall_score(
            validacao[COLUNA_ALVO_FUTURO], previsao, zero_division=0
        ),
        'F1': f1_score(
            validacao[COLUNA_ALVO_FUTURO], previsao, zero_division=0
        ),
        'ACURACIA_BALANCEADA': balanced_accuracy_score(
            validacao[COLUNA_ALVO_FUTURO], previsao
        ),
    })

tabela_limiares = pd.DataFrame(resultados_limiares)
candidatos = tabela_limiares.loc[
    tabela_limiares['PRECISAO'] >= PRECISAO_MINIMA
]

if candidatos.empty:
    raise ValueError('Nenhum limiar atingiu a precisão mínima definida.')

melhor_limiar = candidatos.sort_values(
    ['RECALL', 'F1', 'PRECISAO'], ascending=False
).iloc[0]

limiar_global = float(melhor_limiar['LIMIAR'])

display(melhor_limiar.round(3).rename('validação').to_frame())
print('Limiar escolhido:', round(limiar_global, 3))

## 11. Fazer o teste temporal final

Depois das escolhas, juntamos treino e validação para aproveitar mais dados e treinamos novamente o modelo vencedor. Só então avaliamos nos meses de teste, que ainda não foram usados.

Na matriz de confusão:

- **VN**: previu sem NORM e acertou;
- **FP**: gerou um alerta falso;
- **FN**: deixou de alertar um caso com NORM;
- **VP**: previu NORM e acertou.

In [ ]:
desenvolvimento = pd.concat([treino, validacao], ignore_index=True)

modelo_final = criar_modelos()[nome_modelo]
modelo_final.fit(
    desenvolvimento[FEATURES],
    desenvolvimento[COLUNA_ALVO_FUTURO],
)

y_teste = teste[COLUNA_ALVO_FUTURO]
prob_teste = modelo_final.predict_proba(teste[FEATURES])[:, 1]
prev_teste = (prob_teste >= limiar_global).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_teste, prev_teste, labels=[0, 1]
).ravel()

metricas = pd.Series({
    'AUC ROC': roc_auc_score(y_teste, prob_teste),
    'AUC PR': average_precision_score(y_teste, prob_teste),
    'Referência PR': y_teste.mean(),
    'Acurácia': accuracy_score(y_teste, prev_teste),
    'Acurácia balanceada': balanced_accuracy_score(y_teste, prev_teste),
    'Precisão': precision_score(y_teste, prev_teste, zero_division=0),
    'Recall': recall_score(y_teste, prev_teste, zero_division=0),
    'F1': f1_score(y_teste, prev_teste, zero_division=0),
    'VN': tn,
    'FP': fp,
    'FN': fn,
    'VP': tp,
})

display(metricas.round(3).rename('teste final').to_frame())
print(classification_report(y_teste, prev_teste, digits=3, zero_division=0))

fig, eixos = plt.subplots(1, 2, figsize=(13, 5))

curva_roc = RocCurveDisplay.from_predictions(
    y_teste, prob_teste, name=nome_modelo, ax=eixos[0]
)
curva_roc.line_.set_color('tab:blue')
eixos[0].plot([0, 1], [0, 1], '--', color='gray')

ConfusionMatrixDisplay.from_predictions(
    y_teste,
    prev_teste,
    display_labels=['Sem NORM', 'Com NORM'],
    cmap='Blues',
    ax=eixos[1],
)
eixos[1].set_title(f'Limiar global = {limiar_global:.2f}')

plt.tight_layout()
plt.show()

## 12. Ranking das variáveis

A importância indica quanto cada variável ajudou o Random Forest a tomar suas decisões. Quanto maior o valor, maior foi sua participação no modelo. Importância não significa necessariamente causa do NORM.

In [ ]:
modelo_interno = modelo_final.named_steps['modelo']

ranking_variaveis = pd.DataFrame({
    'VARIAVEL': FEATURES,
    'IMPORTANCIA': modelo_interno.feature_importances_,
})

ranking_variaveis = (
    ranking_variaveis
    .sort_values('IMPORTANCIA', ascending=False)
    .reset_index(drop=True)
)
ranking_variaveis.index = ranking_variaveis.index + 1
ranking_variaveis.index.name = 'POSICAO'

display(ranking_variaveis.round(4))

plt.figure(figsize=(10, 7))
plt.barh(
    ranking_variaveis['VARIAVEL'][::-1],
    ranking_variaveis['IMPORTANCIA'][::-1],
    color='tab:blue',
)
plt.title('Importância das variáveis no modelo final')
plt.xlabel('Importância')
plt.ylabel('Variável')
plt.tight_layout()
plt.show()


## 13. Analisar uma plataforma

Altere apenas `PLATAFORMA_ANALISE`, no início do notebook, para estudar outra plataforma. O gráfico mostra a probabilidade prevista, os resultados reais disponíveis, o limite de alerta e o início do período de teste.

In [ ]:
dados_plataforma = dados.loc[
    dados[COLUNA_PLATAFORMA].eq(PLATAFORMA_ANALISE) & quimica_valida
].copy()

if dados_plataforma.empty:
    print('Não há dados válidos para a plataforma:', PLATAFORMA_ANALISE)
else:
    dados_plataforma['PROB_NORM'] = modelo_final.predict_proba(
        dados_plataforma[FEATURES]
    )[:, 1]
    dados_plataforma['PREVISAO_NORM'] = (
        dados_plataforma['PROB_NORM'] >= limiar_global
    ).astype(int)

    teste_plataforma = dados_plataforma.loc[
        (dados_plataforma['Mes'] > corte_validacao) &
        dados_plataforma[COLUNA_ALVO_FUTURO].notna()
    ].copy()

    print('Plataforma:', PLATAFORMA_ANALISE)
    print('Meses avaliáveis no teste:', len(teste_plataforma))

    if not teste_plataforma.empty:
        y_real = teste_plataforma[COLUNA_ALVO_FUTURO].astype(int)
        y_previsto = teste_plataforma['PREVISAO_NORM']
        tn, fp, fn, tp = confusion_matrix(
            y_real, y_previsto, labels=[0, 1]
        ).ravel()

        resultado = {
            'Acurácia': accuracy_score(y_real, y_previsto),
            'Precisão': precision_score(y_real, y_previsto, zero_division=0),
            'Recall': recall_score(y_real, y_previsto, zero_division=0),
            'F1': f1_score(y_real, y_previsto, zero_division=0),
            'VN': tn,
            'FP': fp,
            'FN': fn,
            'VP': tp,
        }

        # AUC só pode ser calculada quando existem exemplos das duas classes.
        if y_real.nunique() == 2:
            resultado['AUC ROC'] = roc_auc_score(
                y_real, teste_plataforma['PROB_NORM']
            )
            resultado['AUC PR'] = average_precision_score(
                y_real, teste_plataforma['PROB_NORM']
            )

        display(
            pd.Series(resultado)
            .round(3)
            .rename(PLATAFORMA_ANALISE)
            .to_frame()
        )
        tabela_resultados = teste_plataforma[[
            'Mes_alvo', COLUNA_ALVO_FUTURO, 'PROB_NORM', 'PREVISAO_NORM'
        ]].copy()
        tabela_resultados[['PROB_NORM']] = (
            tabela_resultados[['PROB_NORM']].round(3)
        )
        display(tabela_resultados)

    fig, eixo = plt.subplots(figsize=(14, 6))
    eixo.plot(
        dados_plataforma['Mes_alvo'],
        dados_plataforma['PROB_NORM'],
        color='tab:blue',
        marker='.',
        label='Probabilidade prevista um mês antes',
    )

    sem_norm = dados_plataforma.loc[
        dados_plataforma[COLUNA_ALVO_FUTURO].eq(0)
    ]
    com_norm = dados_plataforma.loc[
        dados_plataforma[COLUNA_ALVO_FUTURO].eq(1)
    ]

    eixo.scatter(
        sem_norm['Mes_alvo'], sem_norm[COLUNA_ALVO_FUTURO],
        color='tab:green', s=45, label='Real: sem NORM'
    )
    eixo.scatter(
        com_norm['Mes_alvo'], com_norm[COLUNA_ALVO_FUTURO],
        color='tab:red', s=45, label='Real: com NORM'
    )
    eixo.axhline(
        limiar_global, color='black', linestyle='--',
        label=f'Limiar global = {limiar_global:.2f}'
    )
    eixo.axvline(
        corte_validacao + pd.offsets.MonthBegin(1),
        color='tab:orange', linestyle='--', label='Início do teste'
    )
    eixo.set(
        title=f'Previsão de NORM - {PLATAFORMA_ANALISE}',
        xlabel='Mês previsto',
        ylabel='Probabilidade',
        ylim=(-0.05, 1.05),
    )
    eixo.grid(alpha=0.25)
    eixo.legend()
    plt.show()

## 14. Salvar o modelo

Por fim, salvamos o modelo e todas as informações necessárias para reutilizá-lo sem repetir o treinamento.

In [ ]:
pasta_modelos = PASTA_PROJETO / 'outputs' / 'models'
pasta_modelos.mkdir(parents=True, exist_ok=True)

artefato = {
    'modelo': modelo_final,
    'features': FEATURES,
    'alvo': COLUNA_ALVO_FUTURO,
    'horizonte_meses': 1,
    'limiar': limiar_global,
    'precisao_minima_validacao': PRECISAO_MINIMA,
    'modelo_selecionado': nome_modelo,
    'fim_treino': corte_treino,
    'fim_validacao': corte_validacao,
}

caminho_modelo = pasta_modelos / 'modelo_norm_proximo_mes.joblib'
joblib.dump(artefato, caminho_modelo)

print('Modelo salvo em:', caminho_modelo)